# 05 — Training the Attn-LOB Pretrain Head

Actually trains `paper_replication.models.AttnLOB` on the mid-price direction pretrain task (paper Table I), using `paper_replication.training`. Builds on notebook 04 (architecture-only, no training) and notebook 02 (the feature-engineering pipeline that produces the dataset).

This is a short demonstration run (a handful of epochs), not a full replication-grade training run — the paper doesn't state its epoch count/schedule, and matching Table I's precision/recall/F1 numbers exactly would need much more tuning across more epochs. What this notebook does show: a complete, working train -> validate -> checkpoint -> reload cycle on real data, with a held-out test set never touched until the final evaluation cell.

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
from __future__ import annotations

import pathlib
import time

import matplotlib.pyplot as plt
import torch

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split
from paper_replication.models import AttnLOB, AttnLOBConfig
from paper_replication.training import (
    TrainConfig,
    evaluate,
    load_checkpoint,
    train_model,
)
from paper_replication.training.dataset import AttnLOBTorchDataset

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"
CHECKPOINT_PATH = pathlib.Path("../data/processed/attn_lob_checkpoint.pt")

torch.manual_seed(0)

## Build the dataset and split it chronologically

Same pipeline as notebook 02/04: default `FeatureConfig` (proxy OSI, no ticks file touched), train/val/test split in time order so validation never sees the future.

In [ ]:
dataset = build_feature_dataset(LOB_PATH, FeatureConfig())
splits = chronological_split(dataset)

for name, split in splits.items():
    print(f"{name:>5}: n={len(split)}")

## Train

`train_model` trains in place, tracks the lowest-validation-loss epoch (not necessarily the last one), restores those weights into `model`, and saves them to `CHECKPOINT_PATH`. `TrainConfig`'s defaults set `epochs=50` as an upper bound but `patience=5`: training stops after 5 consecutive epochs with no val_loss improvement, so it won't actually run anywhere near 50. `device` also defaults automatically -- CUDA if a GPU is visible, else CPU.

In [ ]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

model = AttnLOB(AttnLOBConfig())
train_config = TrainConfig(
    batch_size=256,
    learning_rate=1e-3,
    seed=0,
    checkpoint_path=str(CHECKPOINT_PATH),
    # epochs=50 (default) is just an upper bound; patience=5 (default) stops
    # training once val_loss hasn't improved for 5 straight epochs.
    # device left unset -> TrainConfig defaults to "cuda" if a GPU is visible.
)
print(f"device = {train_config.device}")

start = time.time()
history = train_model(
    model, splits["train"], splits["val"], train_config, num_classes=3
)
elapsed = time.time() - start

print(f"ran {len(history)}/{train_config.epochs} epochs (early stop) in {elapsed:.1f}s")
for record in history:
    print(
        f"epoch {int(record['epoch']):>2}  "
        f"train_loss={record['train_loss']:.4f}  "
        f"val_loss={record['val_loss']:.4f}  "
        f"val_acc={record['accuracy']:.3f}  "
        f"val_f1={record['f1']:.3f}"
    )

## Loss curves

In [ ]:
epochs = [r["epoch"] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, [r["train_loss"] for r in history], label="train")
axes[0].plot(epochs, [r["val_loss"] for r in history], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(epochs, [r["accuracy"] for r in history], label="accuracy")
axes[1].plot(epochs, [r["f1"] for r in history], label="macro F1")
axes[1].axhline(1 / 3, color="gray", linestyle="--", linewidth=1, label="chance (1/3)")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].set_title("Validation accuracy / F1")

fig.tight_layout()
plt.show()

## Evaluate on the held-out test split

`splits["test"]` (late-October data) was never touched during training or the loss curves above — this is the first time the model sees it. Loosely comparable in spirit to paper Table I's Attn-LOB row, but not a like-for-like number: different asset (BTC/USDT vs. their equities), a ~10s snapshot grid instead of raw events, an auto-calibrated label threshold, and 8 demo epochs instead of a tuned training run.

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    AttnLOBTorchDataset(splits["test"]), batch_size=256, shuffle=False
)
test_metrics = evaluate(
    model, test_loader, torch.device(train_config.device), num_classes=3
)

for key, value in test_metrics.items():
    print(f"{key:>10} = {value:.4f}")

## Reload the checkpoint

Confirms `CHECKPOINT_PATH` actually holds a usable model: load it into a fresh `AttnLOB` instance (no need to know the config in advance -- it's stored in the checkpoint) and check its output matches `model`'s.

In [ ]:
reloaded = load_checkpoint(str(CHECKPOINT_PATH), map_location=train_config.device)
reloaded.eval()
model.eval()

x = torch.from_numpy(splits["test"].lob_state[:8]).float().to(train_config.device)
with torch.no_grad():
    original_out = model(x)
    reloaded_out = reloaded(x)

print(
    f"checkpoint file  = {CHECKPOINT_PATH}  ({CHECKPOINT_PATH.stat().st_size / 1024:.0f} KB)"
)
print(f"outputs match    = {bool(torch.allclose(original_out, reloaded_out))}")

## Summary

`data/processed/attn_lob_checkpoint.pt` now holds real, trained weights that `load_checkpoint()` can restore into a fresh `AttnLOB` anywhere else in the codebase -- notebook, script, or a future RL-phase training run reusing `forward_features()` as a pretrained backbone.

Val_loss bottomed out at epoch 3 (0.980), then 5 consecutive non-improving epochs (4-8) tripped `patience=5`, stopping the run at epoch 8. `train_model` restored epoch 3's weights, which is what got evaluated above: **49% test accuracy / 0.495 F1**, above the 33% chance floor, on data the model never saw during training.

This is still a short, untuned run -- patience=5 with no learning-rate schedule or regularization. Natural next steps if you want to push accuracy further: weight_decay or dropout to delay overfitting past epoch 3, a learning-rate schedule, and comparing against the paper's other pretrain baselines (FC-LOB, Conv-LOB, DeepLOB) on the same split -- see notebook 06 for a hyperparameter search and notebook 07 for the baseline comparison.